# Lab 4: Geospatial Python Libraries - Fiona and Pandas
#### Interactive Jupyter Notebook

This notebook provides an introduction to two <b>Python libraries</b> for working with data: <b>Fiona</b>, for reading and writing geospatial vector data, and <b>Pandas</b>, for working with tables of data.

## Notebook Outline:
- [Introduction](#intro)
- [Check Libraries](#run_once)
- [Setup](#setup) (run this first!)
- [Fiona](#fiona)
    - [Import Data](#import_data_fiona)
    - [Open a File](#open_file_fiona)
    - [Check and Display Metadata](#display_fiona)
    - [Close File](#close_file_fiona)
    - [Examples in Fiona](#read_fiona)
    - [Exploring Vector Data in Fiona](#explore_fiona)
    - [Mapping Data](#map_fiona)
- [Pandas](#pandas)
    - [Introducing the Series](#series_intro)
        - [Array-like Behavior](#series_as_array)
        - [Dictionary-like Behavior](#series_as_dict)
    - [Introducing the DataFrame](#DataFrame_intro)
        - [From Series to DataFrame](#series_to_DataFrame)
        - [Manipulating a DataFrame](#manip_DataFrame)
    - [Selecting Data in a DataFrame](#select_data)
        - [Select by Label](#select_label)
        - [Select by (only) Position](#select_pos)
        - [Select by Condition](#select_cond)
        - [Grouping and Counting](#group_count)
- [Simple Plotting](#simple_plot)

<a id='intro'></a>
## Introduction

<b>Fiona</b> is a minimalist python package for reading (and writing) vector data in python. Fiona provides python objects (e.g. a dictionary for each record) to geospatial data in various formats. Newer versions of GeoPandas read files with a library called pyogrio by default, but Fiona is still useful when you want to read features one at a time, as we do here.


Useful documentation for these libraries can be found here:

>Fiona https://fiona.readthedocs.io/en/stable/
<br>Pandas https://pandas.pydata.org/docs/user_guide/index.html

<a id='run_once'></a>
## Check Libraries
This cell checks that the libraries this notebook needs are installed, and installs any that are missing. If it installs anything, choose <b>Kernel -> Restart Kernel</b>, then continue with Setup.

In [ ]:
import importlib.util

missing = [p for p in ['fiona', 'geopandas', 'pyproj']
           if importlib.util.find_spec(p) is None]
if missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
    print('Installed:', ', '.join(missing), '- now choose Kernel -> Restart Kernel, then run Setup.')
else:
    print('All libraries are available.')

<a id='setup'></a>
## Setup
Run this cell for the rest of the notebook to work!

In [ ]:
# import required libraries

%matplotlib inline
import os

# use Shapely 2.0 instead of PyGEOS in GeoPandas (must come before importing geopandas)
os.environ['USE_PYGEOS'] = '0'

import fiona
# pretty printing - makes some kinds of text output easier to read
import pprint
import IPython
from matplotlib import pyplot as plt
from pyproj import CRS

import pandas as pd
import geopandas as gpd

print('Setup complete.')

<a id='fiona'></a>
## Fiona

<a id='import_data_fiona'></a>
### Import Data
We will be using the New York City borough boundaries for parts of this lab. The data comes from the NYC Department of City Planning and is in the `data/nybb` folder next to this notebook.

<a id='open_file_fiona'></a>
### Open a File

Load the input_file with Fiona in python.

* What type of geometries are in each feature?
* What properties are on each feature?

(Note: each feature is a dictionary.)

In [ ]:
input_file = 'data/nybb/nybb.shp'
if os.path.exists(input_file):
    print('Input file:', input_file)
else:
    print('Could not find the data. Make sure this notebook is in the ggis407-lab4 folder, next to the data folder.')

In [ ]:
boroughs = fiona.open(input_file)

<a id='display_fiona'></a>
### Check and Display Metadata

When a property or function returns values that Jupyter understands, it will print them
to the output region of the cell in a more human readable form. This can mean pretty-printing
text, showing tabular data as HTML tables, or showing graphical data (like maps) as images.
To learn about an object, you can simply put it on a line by itself, rather than explicitly telling
Python to print it. This only works interactively; you woudn’t want to do this in a script.

In [ ]:
# This will pretty-print the dictionary returned by the
# metadata property of `boroughs`
print("The meta property returns an object of type: ", 
      type(boroughs.meta))
boroughs.meta

In [ ]:
boroughs.meta['schema']

In [ ]:
# The 'crs' value tells us the coordinate reference system of the data.
# Depending on your Fiona version this is shown as a dictionary or as a CRS object.
boroughs.meta['crs']

In [ ]:
# Either way, pyproj can tell us the EPSG code and the name of the CRS
crs = CRS.from_user_input(boroughs.crs_wkt)
print("EPSG code:", crs.to_epsg())
print("Name:", crs.name)

<a id='close_file_fiona'></a>
### Close File
When you open a file, you need to close it when you’re done, or other programs may think it is
still in use. Not explicitly closing a file may also prevent some of the changes you intend to write
to it from being written.

In [ ]:
boroughs.close()

In [ ]:
# Check if the file is closed
boroughs.closed

<a id='read_fiona'></a>
### Examples in Fiona
#### Using a with block
We’ll use the with block, which automatically closes the file when execution leaves the block
of code. That avoids potential problems from forgetting to close the file, or from having an error
prevent the code from reaching the close() command.

In [ ]:
with fiona.open(input_file) as src:
    # Schema
    print("Schema of the shapefile fields: ")
    pprint.pprint(src.schema)
    print()
    # Get first record
    record = next(iter(src))
    print("Values of the first record: ")
    pprint.pprint(dict(record['properties']))

#### Pretty display of Information

In [ ]:
with fiona.open(input_file) as src:
    #Get all record
    for record in src:
        print("Values of the record: ")
        pprint.pprint(dict(record['properties']))

In [ ]:
# Speaking of making the output easier to read, it's
# possible to construct tables with Markdown or HTML
# Later, we'll use a different Python library that
# happens to do the nice formatting automatically.
with fiona.open(input_file) as src:
    # make header row out of field names
    values="|"
    for field in src.schema['properties']:
        values += field
        values += "|"
    values += '\n|'
    # separates header row
    for i in range(len(src.schema['properties'])):
        values += "---|"
    values += "\n"
    # rows for each record
    for record in src:
        values += "|"
        for field in record['properties']:
            values += str(record['properties'][field])
            values += '|'
        values += '\n'
    print("Markdown text for displaying as table: \n",
          values, sep="\n") #\n is code for new line
    print("\nSame text presented as table: ")
    IPython.display.display_markdown(values, raw=True)

That took a lot of code. GeoPandas, which you'll use in the next notebook, reads the same file straight into a table and formats it for you. Here we drop the geometry column so only the properties are shown:

In [ ]:
# GeoPandas does the table formatting for us
gpd.read_file(input_file).drop(columns='geometry')

#### Calculate the Total Area

In [ ]:
total_area=0
with fiona.open(input_file) as src:
    for record in src:
        total_area+=record['properties']['shape_area']
print("Total Area is ", total_area)

<a id='explore_fiona'></a>
### Exploring Vector Data in Fiona

In [ ]:
# We have a `record` variable from earlier.
# What exactly is that? In older versions of Fiona it is a
# plain Python dictionary; in newer versions it is a Feature
# object that you can use just like a dictionary.
type(record)

In [ ]:
list(record.keys())

In [ ]:
# We saw what was in 'properties' in our 
# schema check earlier.
# Which borough is this record?
record['properties']['boro_name']

In [ ]:
record['geometry']['type']

In [ ]:
# When we checked the schema earlier,
# we saw that the geometry type was Polygon.
# What can we do with the geometry?
print("The type of object for the record's geometry property is: ",
      type(record['geometry']))
print()
print("The keys of the dictionary are: ", list(record['geometry'].keys()))

In [ ]:
# What are in those coordinates?
record['geometry']['coordinates']

In [ ]:
record['geometry']['type']

In [ ]:
# That's a lot of values. It looks like there
# are three levels of nested lists.
# If the first level of the list are the individual
# polygons that make up this borough, how many are there?
dim1 = len(record['geometry']['coordinates'])
print("The", record['properties']['boro_name'],
      "feature has", dim1, "polygons.")

In [ ]:
# What's in the next level?
dim2 = []
for i in range(dim1):
    dim2 += [len(record['geometry']['coordinates'][i])]
print("The number of lists for each multi-polygon is:\n", dim2)

In [ ]:
# Well, that makes it easier to check the third level.
# Write a for loop that checks how many points
# belong to each feature.
# Remember, you know that for the second level of the
# list, there is only one item (another list)
dim3 = []
for i in range(dim1):
    dim3 += [len(record['geometry']['coordinates'][i][0])]
print("The number of points for each polygon is:\n", dim3)

<a id='map_fiona'></a>
### Mapping Data

In [ ]:
plt.figure(figsize=(4, 4))
for coords in record['geometry']['coordinates']:
    # We are one level deep in the list
    # here '*' is an unpack operator
    x, y = zip(*coords[0])
    # The [0] opens the second level of the list
    # (this level having only one member each)
    # The unpack operator gets the contents of the
    # list out of the list, and sends to the zip function
    plt.plot(x, y)

<a id='pandas'></a>
## Pandas

<b>Pandas</b> is a Python library for data analysis, featuring easy IO, powerful indexing, database operations, and categorical data support. Two primary data structures in Pandas are <b>`Series`</b> and <b>`DataFrame`</b>.

Useful documentation for these libraries can be found here:

>Pandas https://pandas.pydata.org/docs/ <br>
Intro to data structures (Series and DataFrame) https://pandas.pydata.org/docs/user_guide/dsintro.html

<a id='series_intro'></a>
### Introducing the Series

A `Series` in Pandas is a <b>one-dimensional labeled array</b> capable of holding a wide variety of data. This array also has an <b>index</b>, which is like a label for each of the values in the array. The basic method to create a `Series` is to call:

> pd.Series(data, index=index)

Here, data can be many different things, such as:

- a Python dictionary
- a numpy array
- a scalar value (like `5`)

Anything like an array, iterable, a dictionary, or a scalar value, will work for data contents!

Basically, a `Series` is a list with an index!

Here are some examples of constructing a <b>`Series`</b>:

In [ ]:
# create a basic series

pd.Series([0,1,2], index=['a', 'b', 'c'])

Note that index values are on the left.

In [ ]:
# create a series using a dictionary of values and index values

pd.Series({'a' : 0, 'b' : 1, 'c' : 2})

In [ ]:
# scalar values will be repeated to be the length of the index

pd.Series(5, index=['a', 'b', 'c'])

In [ ]:
# but the opposite does not work
# this cell will fail to run!

pd.Series([5, 5, 5], index=['a'])

In [ ]:
# if no index is provided, data values will be used as an index

pd.Series([0,1,2])

Once you have a series object you can access the <b>data</b> and <b>index values</b> directly:

In [ ]:
# see type of series

myseries = pd.Series({'a' : 0, 'b' : 1, 'c' : 2})
type(myseries)

In [ ]:
# list series values

list(myseries.values)

In [ ]:
# list series index values

list(myseries.index)

<a id='series_as_array'></a>
#### Array-like Behavior

You can access elements of a <b>`Series`</b> by <b>position</b>, just like for regular lists and tuples in Python, using <b>`.iloc`</b>. Remember that Python indices start with 0.

Here are a few examples:

In [ ]:
# return the first element (by position)

myseries.iloc[0]

In [ ]:
# select everything up until position 2 (the third element)

myseries.iloc[:2]

In [ ]:
# find the median value

myseries.median()

Operations can be performed element-wise for the `series`.

In [ ]:
# see which elements are greater than the medium

myseries > myseries.median()

Use `any` or `all` to check if a comparison is true for the series as a whole.

In [ ]:
# are any elements greater than the medium value?

any(myseries > myseries.median())

In [ ]:
# are all elements greater than the medium value?

all(myseries > myseries.median())

You can also select values using a logical index series.

In [ ]:
# select by greater than median

myseries[myseries > myseries.median()]

In [ ]:
myseries

You can also pass a list of positions to select series elements out of order.

In [ ]:
# select by list of positions

myseries.iloc[[2, 1, 1]]

Other operations are performed element-wise too. For example:

In [ ]:
# add series values to self

myseries + myseries

In [ ]:
# raise to the 3rd power

myseries ** 3

<a id='series_as_dict'></a>
#### Dictionary-like Behavior

Since a <b>`Series`</b> has an `index`, you can also work with a `Series` as if it were a Python dictionary:

In [ ]:
# find value with index 'a'

myseries['a']

In [ ]:
# set value of 'd'

myseries['d'] = 5
myseries

In [ ]:
# see if index value in series

'f' in myseries

In [ ]:
# see if index value in series

'c' in myseries

<b>'Naming' a series</b>

A `Series` has an optional <b>`name`</b> parameter:

In [ ]:
# see current name (not assigned)

print(myseries.name)

If you name a `Series`, the name will show up in the information at the bottom:

In [ ]:
# set name

myseries=myseries.rename('stock_price')

In [ ]:
# see current name (not assigned)

print(myseries.name)

<a id='DataFrame_intro'></a>
### Introducing the DataFrame

What happens when a `Series` gains a dimension? It becomes a <b>`DataFrame`</b>!

Let's load a table of the 1,000 largest U.S. cities (2013 Census Bureau population estimates) to see how to check out a `DataFrame`. The data is a zipped shapefile in the `data` folder, and GeoPandas can read it directly from the zip file. The `head` function is useful to see the first few rows of a `DataFrame`.

In [ ]:
# import cities database

towns_gpd = gpd.read_file('zip://data/us_cities.zip')
towns_table = pd.DataFrame(towns_gpd)

In [ ]:
# see start of DataFrame

towns_table.head()

Similarly, the `tail` function can be used to check the end:

In [ ]:
# see end of DataFrame

towns_table.tail()

Like a `Series`, a `DataFrame` has an <b>index</b>. This one is a numeric range.

In [ ]:
# see index

towns_table.index

You can also list the `DataFrame` <b>column names</b>:

In [ ]:
# see column names

towns_table.columns

And it is often useful to dig into its <b>values</b>:

In [ ]:
# see values

towns_table.values

It can be useful to double check the size of the values array. The `index` told us how many rows there are; now we can count the columns (by a few different methods).

In [ ]:
# see the size of the values array

towns_table.values.shape

In [ ]:
# count columns

print(len(towns_table.values[0]))
print(len(towns_table.columns))

Using index selection we can see all values in one row.

In [ ]:
# see one row

towns_table.values[5]

We can use the double index method to find a value within a particular and row and column:

In [ ]:
# find specific value (row 5, column 1)

towns_table.values[5][1]

Sometimes it is useful to know summary information about a `DataFrame`. The `describe` makes this clear:

In [ ]:
# summarize DataFrame

towns_table.describe()

Transposing a `DataFrame` can also come in handy:

In [ ]:
# transpose DataFrame

towns_table.T

It is also possible to <b>sort</b> a `DataFrame` by values in a particular column:

In [ ]:
# sort DataFrame by column

towns_table.sort_values(by='POPULATION')

<a id='series_to_DataFrame'></a>
#### From Series to DataFrame

Now that we are slightly more familiar with the `DataFrame`, we can go into more detail. Technically, a `DataFrame` is implemented as a list of named `Series` all <b>with the same index values<b>.

In [ ]:
# construct a DataFrame

myDataFrame = pd.DataFrame(
    {'one' : pd.Series([1., 2., 3.], index=['a', 'b', 'c']),
     'two' : pd.Series([1., 2., 3., 4.], index=['a', 'b', 'c', 'd'])})

myDataFrame

Remember the `name` attribute for a `Series`? Since each column in a `DataFrame` is a `Series`, each column name is equivalent to the name of the `Series` for that column.

In [ ]:
# show DataFrame column names

myDataFrame.columns

<a id='manip_DataFrame'></a>
#### Manipulating a DataFrame

Columns in a `DataFrame` can be freely selected, added, deleted (as with the "dictionary" aspect of `Series`):

In [ ]:
# create new column from existing columns

myDataFrame['three'] = myDataFrame['one'] + myDataFrame['two']
myDataFrame

Columns in a `DataFrame` do not have to all be the same data type (as we saw with the towns database):

In [ ]:
# create new column from comparison

myDataFrame['flag'] = myDataFrame['one'] > 2
myDataFrame

It is simple to <b>delete</b> a column:

In [ ]:
# delete column

del myDataFrame['two']
myDataFrame

Scalar values assigned to a column repeat to fit the length of the `DataFrame`.

In [ ]:
# add column

myDataFrame['four'] = 4
myDataFrame

You can also add a new row to the `DataFrame` by building a one-row `DataFrame` and combining the two with `pd.concat`.

In [ ]:
# add row to DataFrame

new_row = pd.DataFrame([{'one': 3, 'three': 2.0, 'flag': False}])
pd.concat([myDataFrame, new_row], ignore_index=True)

Note that the changes won't affect the original `DataFrame` unless you re-assign it.

In [ ]:
myDataFrame

In [ ]:
# add row to DataFrame and keep the result

myDataFrame2 = pd.concat([myDataFrame, new_row], ignore_index=True)

In [ ]:
myDataFrame2

Don't add rows one at a time too often, as `pd.concat` makes a full copy of the data, so constantly using it can create a significant performance hit. A better way is to include all rows while creating the DataFrame.

You can also <b>drop</b> rows and columns:

In [ ]:
# drop rows

myDataFrame.drop(['a','c'])

`axis` specifies whether you are referring to rows or columns:

In [ ]:
# drop columns

myDataFrame.drop(['one','four'], axis=1)

<a id='select_data'></a>
### Selecting Data in a DataFrame

<b>Selecting data</b> is a common task when dealing with a `DataFrames`.

<a id='select_label'></a>
#### Select by Label

As seen before, we can select data using `label` values.

In [ ]:
# select column by label

towns_table['POPULATION']

We can select just a few rows with index slicing:

In [ ]:
# select rows

towns_table[0:3]

We can combine row selection using indices and selecting columns using labels using the <b>`loc`</b> function:

In [ ]:
# select rows and columns

towns_table.loc[0:2,['POPULATION','CITY_NAME']]

In [ ]:
# select row and columns

towns_table.loc[3,['STATE_NAME','CITY_NAME','POPULATION']]

In [ ]:
# select specific value

towns_table.loc[3,'POPULATION']

<a id='select_pos'></a>
#### Select by (only) Position

Column label values are not necessary - you can select values using just indices with the <b>`iloc`</b> function. Here are some examples:

In [ ]:
# select row

towns_table.iloc[3]

In [ ]:
# select some columns from row

towns_table.iloc[3,0:3]

In [ ]:
# select some columns from all rows

towns_table.iloc[:,0:3]

In [ ]:
# select rows and columns

towns_table.iloc[:2,:2]

<a id='select_cond'></a>
#### Select by Condition

We can use conditional operators to select `DataFrame` slices that meet specific conditions. Here are examples:

In [ ]:
# select by condition

towns_table['POPULATION']>1000000

In [ ]:
# select values by condition: cities with more than one million people

towns_table[towns_table['POPULATION']>1000000]

In [ ]:
# create new database from selection of 20 cities

simp_db = towns_table.loc[0:20,['CITY_NAME','POPULATION']]

In [ ]:
simp_db

<a id='group_count'></a>
#### Grouping and Counting
Often you want to summarize a table by category instead of looking at individual rows. `value_counts` counts how many rows have each value in a column, and `groupby` splits the table into groups (here, one per state) so you can calculate something for each group, such as a sum or a mean.

In [ ]:
# how many of the 1,000 largest cities are in each state?
towns_table['STATE_NAME'].value_counts().head(10)

In [ ]:
# total population of those cities, by state
towns_table.groupby('STATE_NAME')['POPULATION'].sum().sort_values(ascending=False).head(10)

<a id='simple_plot'></a>
## Simple Plotting

It can be quick to get a simple `DataFrame` plot up, but finessing the plot can take some time. Our first plot isn't that helpful...

In [ ]:
# basic plot

simp_db.plot();

Let us see if we can do better. First we see if we can take a transpose and plot it.

In [ ]:
# transpose

simp_db.T

In [ ]:
# this doesn't work! (this cell will show an error)

simp_db.T.plot()

That did not work: after transposing, every column mixes a town name and a number, so Pandas finds no numeric data to plot. Let's get the town names into the header and have the table contain just population.

In [ ]:
# rename columns with town names

simp_db2 = simp_db.T.rename(columns=simp_db.T.iloc[0])
simp_db2

In [ ]:
# now drop town column

simp_db3 = simp_db2.drop(['CITY_NAME'])
simp_db3

Now we have town names in our header and population values in the table. Because the values came from a mixed column, we convert them to numbers with `astype(int)` before plotting. We get a different chart type, and it works better.

In [ ]:
# bar plot

simp_db3.astype(int).plot(kind='bar',figsize=(14,12));

A shortcut: instead of transposing and renaming, you can tell Pandas which column to use for the x-axis and which to plot:

In [ ]:
# bar plot straight from simp_db

simp_db.plot.bar(x='CITY_NAME', y='POPULATION', figsize=(14,6));

In [ ]:
# Practice what you have learned about Fiona and Pandas
# Feel free to modify any code above
# Try these: 
#     What is the average city population in each state? Which state’s large cities are biggest on average? (Hint: change .sum() in the groupby cell.)
#     Make a new table with only the cities in Illinois, sorted by population. How many are there, and which is the smallest?